# Урок 20. Классификация на практике: соседи и деревья

**Дата:** 04.02.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 19 (дерево решений, `fit / predict`, train / test, accuracy).

На прошлом уроке мы обучили первую модель — дерево решений. Сегодня познакомимся со вторым алгоритмом — **методом k ближайших соседей** — и сравним модели **честно**, на test. Узнаем, почему одной модели важен масштаб признаков, а другой — нет, как превратить категории в числа и как разбирать ошибки модели.

## Цели урока

К концу занятия ты сможешь:

1. Объяснять идею метода k ближайших соседей (kNN).
2. Понимать, почему kNN чувствителен к **масштабу признаков**, и масштабировать их честно.
3. Сравнивать две модели на одном и том же test.
4. Узнавать, какие признаки важны для дерева.
5. Находить и разбирать ошибки модели.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–15 | Метод ближайших соседей |
| 15–25 | Масштаб признаков |
| 25–35 | Сравнение моделей, важность признаков, разбор ошибок |
| 35–40 | Категориальные признаки |
| 40–55 | Практика |
| 55–60 | Итог и домашнее задание |

## 1. Данные

Ученики те же, но теперь у нас есть ещё один столбец — **шаги в день** по данным фитнес-браслета. Мы заложили, что на результат он **не влияет** — посмотрим, как с ним справятся разные модели.

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def make_students(n, seed):
    """Учебная таблица учеников: как на уроке 12 + шаги в день."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "шаги": random.randint(3000, 15000),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(240, seed=20)
df["сдал"] = df["балл"] >= 60

X = df[["часы", "посещаемость", "шаги"]]
y = df["сдал"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("train:", X_train.shape, "| test:", X_test.shape)
print("Доля сдавших в test:", round(y_test.mean(), 2))

## 2. Метод k ближайших соседей

Идея простая: **«скажи мне, кто твои соседи, и я скажу, кто ты»**.

Чтобы сделать предсказание для нового ученика, модель:

1. находит в обучающих данных **k** учеников, которые больше всего на него похожи (ближайших **соседей**);
2. смотрит, сдали ли они;
3. отвечает так, как ответило **большинство** соседей.

«Похожесть» — это **расстояние** между учениками, если представить каждого точкой на графике: по одной оси часы, по другой посещаемость. Ближе точки — похожее ученики.

**Пример.** Ученик А: 6 часов, посещаемость 70. Ученик Б: 9 часов, посещаемость 74. Разница по часам — 3, по посещаемости — 4. Расстояние между точками считают по **теореме Пифагора**, как гипотенузу прямоугольного треугольника с катетами 3 и 4: √(3² + 4²) = √25 = 5. Если признаков больше двух, формула та же — просто под корнем больше слагаемых.

> **Аналогия.** Ты переехал в новый город и хочешь понять, хорошее ли кафе на углу. Спрашиваешь пятерых ближайших соседей: трое говорят «хорошее» — значит, скорее всего, хорошее.

Метод k ближайших соседей по-английски называют **kNN** (*k nearest neighbours*), а `k` — число соседей, которых спрашиваем.

Посмотрим на соседей глазами: обучающие ученики — точки, новый ученик — звезда.

In [ ]:
passed = X_train[y_train]
failed = X_train[~y_train]

plt.figure(figsize=(6, 5))
plt.scatter(passed["часы"], passed["посещаемость"], color="seagreen", alpha=0.6, label="сдал")
plt.scatter(failed["часы"], failed["посещаемость"], color="crimson", alpha=0.6, label="не сдал")
plt.scatter([6], [70], color="black", marker="*", s=300, label="новый ученик")   # s — размер значка
plt.title("Кто соседи нового ученика?")
plt.xlabel("Часов в неделю")
plt.ylabel("Посещаемость, %")
plt.legend()
plt.show()

Какие точки рядом со звездой? Что, по-твоему, скажет kNN?

## 3. kNN в sklearn — и неожиданный провал

Схема та же: создать → `fit` → `predict`. Модель называется `KNeighborsClassifier`, параметр `n_neighbors` — это k.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("kNN, accuracy на test:", round(accuracy_score(y_test, knn.predict(X_test)), 2))

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
print("Дерево, accuracy на test:", round(accuracy_score(y_test, tree.predict(X_test)), 2))

kNN справился **хуже, чем если бы всем отвечал «сдаст»** (в test сдали 72%). Почему?

## 4. Масштаб признаков

kNN измеряет расстояние. Посмотри на разницу масштабов:

| Признак | Диапазон | «Шаг» между учениками |
|---------|----------|------------------------|
| часы | 1–14 | разница 5 часов — огромная |
| посещаемость | 20–100 | разница 30% — большая |
| шаги | 3000–15000 | разница 2000 шагов — обычная |

Вспомни формулу расстояния: разница по шагам 2000 даёт под корнем 2000² = 4 000 000, а разница по часам 5 — всего 25. Для расстояния число 2000 намного больше, чем 5. Получается, что kNN ищет соседей **почти только по шагам** — а шаги на результат не влияют! Признак с большими числами «перекрикивает» остальные.

Решение — **привести признаки к одному масштабу**. Самый простой способ: разделить каждый столбец на его максимум. Тогда все признаки будут от 0 до 1.

**Важно:** максимум берём **только по train** — test мы не подсматриваем, даже ради масштабирования (урок 19). Тот же максимум применяем к test.

In [ ]:
max_train = X_train.max()
print("Максимумы по train:")
print(max_train)

X_train_scaled = X_train / max_train     # каждый столбец делится на СВОЙ максимум
X_test_scaled = X_test / max_train
X_train_scaled.head()

In [ ]:
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
print("kNN после масштабирования, test:", round(accuracy_score(y_test, knn_scaled.predict(X_test_scaled)), 2))

Совсем другое дело! Вывод: **для kNN масштаб признаков критичен**.

А дереву масштаб безразличен: оно задаёт вопросы про **один** признак за раз («часы > 5.5?»), и ему всё равно, в каких единицах этот признак. Это важное практическое различие двух моделей.

## 5. Какие признаки важны для дерева

У обученного дерева есть свойство `feature_importances_` — доля «вклада» каждого признака в решения дерева (сумма равна 1). Подчёркивание в конце имени — договорённость sklearn: так помечают то, что появляется **после** обучения (`fit`).

Это набор чисел без подписей, поэтому соберём из него столбец pandas вручную: `pd.Series(значения, index=названия)` — значения и подписи к ним.

In [ ]:
importance = pd.Series(tree.feature_importances_, index=X.columns)
print(importance.round(2).sort_values(ascending=False))

Дерево почти не пользуется шагами — оно само «отбросило» бесполезный признак. Осторожно с выводами: **важность для модели — не то же самое, что причина в жизни** (вспомни «корреляция — не причина» из урока 12).

## 6. Разбор ошибок

Accuracy — одно число. Инженер смотрит глубже: **на ком** модель ошибается? Соберём таблицу «правда — дерево — соседи» для test и оставим строки, где хоть одна модель ошиблась.

In [ ]:
compare = X_test.copy()
compare["правда"] = y_test
compare["дерево"] = tree.predict(X_test)
compare["соседи"] = knn_scaled.predict(X_test_scaled)

errors = compare[(compare["правда"] != compare["дерево"]) | (compare["правда"] != compare["соседи"])]
print("Строк с ошибкой хотя бы одной модели:", len(errors))
errors

Посмотри на часы у этих учеников: в основном **от 3 до 7** — пограничная зона, где «сдал / не сдал» решает шум. Далеко от границы (2 часа или 12 часов) модели почти не ошибаются. Это нормально: ошибки на границе есть у любой модели. Опасно, когда модель ошибается **систематически** на какой-то группе — об этом урок 24.

## 7. Категориальные признаки

Модели sklearn понимают только числа. Как использовать столбец `кружок` со значениями «Python», «Роботы», «Игры»?

Нельзя просто пронумеровать (Python = 1, Роботы = 2, Игры = 3): модель решит, что «Игры» в три раза «больше» Python, а это бессмыслица. Правильно — сделать для каждого значения **отдельный столбец** «да / нет». Это делает `pd.get_dummies`:

In [ ]:
X_cat = pd.get_dummies(df[["часы", "посещаемость", "кружок"]], columns=["кружок"])
X_cat.head()

Каждый ученик получил три столбца `True / False`, и ровно в одном из них `True`. Такую таблицу уже можно передавать в `fit`. Этот приём называют **one-hot-кодированием** («одна горячая»: одна единица среди нулей).

## 8. Практика

### Практика 1. Подбери k (8 минут)

Обучи kNN на масштабированных данных с `k` = 1, 3, 5, 9, 15, 25. Для каждого выведи accuracy на **train** и на **test**. Что происходит при `k = 1` на train? Почему?

In [ ]:
# Практика 1

<details>
<summary>Решение</summary>

```python
for k in [1, 3, 5, 9, 15, 25]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    acc_train = accuracy_score(y_train, m.predict(X_train_scaled))
    acc_test = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"k = {k:2}: train {acc_train:.2f}, test {acc_test:.2f}")
```
При `k = 1` ближайший сосед любого ученика из train — он сам, поэтому на train ответ всегда верный (1.0). Это та же «зубрёжка», что у глубокого дерева. На test так не получается.
</details>

### Практика 2. Без лишнего признака (6 минут)

Убери из признаков `шаги` (оставь часы и посещаемость), **не масштабируй** и обучи kNN с `k = 5`. Сравни с результатом из раздела 3. Объясни разницу.

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
X2 = df[["часы", "посещаемость"]]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y
)
m2 = KNeighborsClassifier(n_neighbors=5)
m2.fit(X2_train, y2_train)
print(round(accuracy_score(y2_test, m2.predict(X2_test)), 2))
```
Без шагов kNN работает хорошо даже без масштабирования: часы и посещаемость сравнимы по масштабу и оба полезны. Плохой признак с большим масштабом вредил сильнее всего.
</details>

### Практика 3. Два мнения о новом ученике (5 минут)

Новый ученик: 6 часов, посещаемость 70, 8000 шагов (звезда на графике). Что скажут дерево и kNN? Не забудь, что для `knn_scaled` признаки ученика тоже нужно разделить на `max_train`.

In [ ]:
new_student = pd.DataFrame({"часы": [6], "посещаемость": [70], "шаги": [8000]})
# Практика 3

<details>
<summary>Решение</summary>

```python
print("Дерево:", tree.predict(new_student))
print("kNN:", knn_scaled.predict(new_student / max_train))
```
</details>

### Практика 4. Кружок как признак (6 минут)

Обучи два дерева глубины 3 на одном и том же разбиении: на `X_cat` (часы, посещаемость и три столбца кружка) и только на часах и посещаемости. Помог ли кружок? Посмотри на важность признаков. Согласуется ли это с «правилом мира»?

In [ ]:
# Практика 4

<details>
<summary>Решение</summary>

```python
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cat, y, test_size=0.25, random_state=42, stratify=y
)
tree_cat = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_cat.fit(Xc_train, yc_train)
print("С кружком, test:", round(accuracy_score(yc_test, tree_cat.predict(Xc_test)), 2))
print(pd.Series(tree_cat.feature_importances_, index=X_cat.columns).round(2))

cols = ["часы", "посещаемость"]
tree_base = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_base.fit(Xc_train[cols], yc_train)
print("Без кружка, test:", round(accuracy_score(yc_test, tree_base.predict(Xc_test[cols])), 2))
```
Кружок ничего не добавляет: в нашем «правиле мира» он не влияет на балл, и дерево на него почти не смотрит. Сравнивать модели нужно на **одном и том же** разбиении и меняя только одно — иначе непонятно, что дало эффект.
</details>

## Работа с ИИ-помощником

- «Объясни, почему kNN чувствителен к масштабу признаков, а дерево — нет.»
- «Почему при k = 1 accuracy на train равна 1?»
- «Вот таблица ошибок моей модели. Какие закономерности стоит проверить?»

## Мини-глоссарий урока 20

| Термин | Смысл |
|--------|-------|
| **kNN** | метод k ближайших соседей: ответ — как у большинства похожих примеров |
| **k (`n_neighbors`)** | сколько соседей спрашиваем |
| **Расстояние** | мера непохожести объектов по признакам |
| **Масштабирование** | приведение признаков к одному масштабу (например, от 0 до 1) |
| **Важность признака** | вклад признака в решения модели |
| **One-hot-кодирование** | категория → набор столбцов «да / нет» |

## Итог урока

1. kNN отвечает «как большинство похожих»; ему нужны признаки в одном масштабе.
2. Масштабирование и любые другие расчёты делаем по train и применяем к test.
3. Модели сравнивают на одном test, а затем разбирают **ошибки**, а не только accuracy.

## Домашнее задание (20 минут)

Возьми **другое зерно** данных — `make_students(240, seed=7)` — и повтори сравнение: дерево глубины 3 и kNN с масштабированием (k = 5). Какая модель лучше на test? Совпал ли «победитель» с сегодняшним? Напиши вывод в 3 предложения.

In [ ]:
# Домашнее задание

## Самопроверка

1. Как kNN делает предсказание?
2. Почему признак «шаги» испортил kNN, но не дерево?
3. Почему максимум для масштабирования берём только по train?
4. Почему кружки нельзя просто пронумеровать 1, 2, 3?
5. Где чаще всего ошибаются модели в нашей задаче?

**Следующий урок (11.02.2027):** метрики классификации — почему одной accuracy недостаточно.